> **교육 목표** 라벨이 일부(10%)뿐일 때 라벨 없는 데이터를 활용하는 준지도 학습(Self-Training)의 효과를 확인합니다.
>
> **핵심 내용** 학습 라벨 90% 숨기기 → ① 라벨 10%만 ② Self-Training ③ 전체 라벨(상한선) 비교 → 가짜 라벨의 정확도

In [ ]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D3_04 라벨이 10%뿐이라면: 준지도 학습

> **오늘 질문:** 정답이 일부에만 있을 때, 라벨 없는 데이터도 학습에 쓸 수 있을까?

검사는 샘플링으로만 하는 현장을 흉내 냅니다. 학습 데이터의 **라벨 90%를 일부러 숨기고** 세 가지를 비교합니다.

| 실험 | 학습에 쓰는 것 |
|---|---|
| ① 라벨 10%만 | 라벨 있는 기록만 |
| ② Self-Training | 라벨 10% + 라벨 없는 90% |
| ③ 전체 라벨 (상한선) | 모든 라벨 |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비: 3교시 Data Mart 피처 📋

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

df = pd.read_parquet("../../data/factory_table.parquet")
df["temp_diff"] = df["process_temp"] - df["air_temp"]
df["power"] = df["torque"] * df["rpm"]
df["strain"] = df["tool_wear"] * df["torque"]
df["grade_num"] = df["grade"].map({"L": 0, "M": 1, "H": 2})
features = ["air_temp", "process_temp", "rpm", "torque", "tool_wear",
            "vibration", "temp_diff", "power", "strain", "grade_num"]

cut = int(len(df) * 0.8)
X_train, X_test = df[features].iloc[:cut], df[features].iloc[cut:]
y_train, y_test = df["failure"].iloc[:cut], df["failure"].iloc[cut:]

## 1. 라벨 90% 숨기기 ✍️

scikit-learn의 준지도 학습에서는 **라벨이 없는 행을 -1**로 표시합니다.

> 변수 이름: `y_semi`, `labeled`

In [ ]:
# 힌트: 난수 < 0.9 인 행을 -1로

## 2. ① 라벨 10%만으로 학습 ✍️

> 변수 이름: `pred_10`

In [ ]:
# 힌트: 모델.fit(X[labeled], y[labeled])

## 3. ② Self-Training ✍️

확신이 90% 이상인 예측부터 가짜 라벨로 붙이고, 다시 학습하기를 반복합니다.

> 변수 이름: `self_model`, `pred_self`

In [ ]:
# 힌트: SelfTrainingClassifier(기본모델, threshold=0.9).fit(X, 숨긴라벨)

## 4. ③ 전체 라벨 (상한선) ✍️

> 변수 이름: `pred_full`

## 5. 세 실험 비교 📋

In [ ]:
preds = {"① 라벨 10%": pred_10, "② Self-Training": pred_self, "③ 전체 라벨": pred_full}
rows = []
for name, p in preds.items():
    rows.append({"실험": name,
                 "정밀도": precision_score(y_test, p),
                 "재현율": recall_score(y_test, p),
                 "F1": f1_score(y_test, p)})
pd.DataFrame(rows).round(2)

## 6. 가짜 라벨은 얼마나 맞았을까 📋

실습이라 숨긴 정답을 알고 있으므로, Self-Training이 붙인 가짜 라벨을 진짜 정답과 비교해 봅니다.

In [ ]:
pseudo = pd.Series(self_model.transduction_, index=y_train.index)[~labeled]   # 숨긴 행에 붙은 라벨
truth = y_train[~labeled]
added = pseudo != -1
print(f"가짜 라벨이 붙은 행: {added.sum()}건 / 숨긴 행 {len(pseudo)}건")
print(f"  그중 '고장'으로 붙인 것 {(pseudo[added] == 1).sum()}건, 실제 고장 {truth[added].sum()}건")
print(f"  가짜 라벨 정확도 {(pseudo[added] == truth[added]).mean():.1%}")
print(f"숨긴 행 중 실제 고장 {truth.sum()}건")

## 정리 💬

**Q1.** ②는 ①과 ③ 사이 어디에 있나요? 이번 데이터에서 Self-Training이 도움이 됐나요?

> 답:


**Q2.** 가짜 라벨 결과를 보면, 고장으로 붙인 수와 실제 고장 수가 어떻게 다른가요? 현장에서 라벨을 더 확보한다면 어떤 기록을 우선 검사하겠습니까?

> 답:
